[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Boutique_Fisica_Avanzada/CFD_Adveccion_Difusion.ipynb)



# Dinámica de Fluidos Computacional: Ecuación de Advección-Difusión
La base matemática matemática rigurosa detrás de Navier-Stokes y de la dispersión de contaminantes en fluidos es la **Ecuación de Advección-Difusión**.

$$ \frac{\partial C}{\partial t} + \mathbf{u} \cdot \nabla C = D \nabla^2 C $$

- **Término de Advección ($\mathbf{u} \cdot \nabla C$)**: El fluido transporta pasivamente la mancha empujándola inercialmente.
- **Término de Difusión ($D \nabla^2 C$)**: El caos térmico viscoso molecular desenfoca y esparce suavemente los bordes de la mancha.

Aquí resolvemos la Ecuación Diferencial en Derivadas Parciales (EDP) en 2D utilizando el **Método de Diferencias Finitas (FTCS)**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Parámetros del espacio físico y temporal
Nx, Ny = 100, 100
Lx, Ly = 1.0, 1.0
dx, dy = Lx/(Nx-1), Ly/(Ny-1)

# Parámetros físicos del fluido
u = 0.5      # Velocidad de advección en el eje X (Viento hacia la derecha)
v = 0.2      # Velocidad de advección en el eje Y (Viento leve hacia arriba)
D = 0.005    # Coeficiente de Difusión (Viscosidad de dispersión)

# Paso de tiempo de seguridad (Criterio de Courant-Friedrichs-Lewy CFL)
dt = min(0.25 * dx**2 / D, 0.5 * dx / abs(u))

# Inicializar el campo de concentración C (Matriz vacía)
C = np.zeros((Nx, Ny))

# Inyectar una burbuja de alta concentración (Tinte/Contaminante) en el centro
r_cx, r_cy = int(Nx*0.2), int(Ny*0.2)
radius = 5
for i in range(Nx):
    for j in range(Ny):
        if (i - r_cx)**2 + (j - r_cy)**2 < radius**2:
            C[i, j] = 1.0

C_new = np.copy(C)
steps = 100

# Bucle de evolución temporal (Método Forward in Time, Centered in Space - FTCS y Upwind)
for _ in range(steps):
    # Copia del estado actual
    C_old = np.copy(C_new)
    
    # Iterar sobre los nodos internos del grid
    for i in range(1, Nx-1):
        for j in range(1, Ny-1):
            # Difusión discreta (Laplaciano 2D 5-puntos)
            diff_x = (C_old[i+1, j] - 2*C_old[i, j] + C_old[i-1, j]) / dx**2
            diff_y = (C_old[i, j+1] - 2*C_old[i, j] + C_old[i, j-1]) / dy**2
            
            # Advección discreta (Esquema Upwind para estabilidad inercial)
            adv_x = u * (C_old[i, j] - C_old[i-1, j]) / dx
            adv_y = v * (C_old[i, j] - C_old[i, j-1]) / dy
            
            # Actualización del paso temporal
            C_new[i, j] = C_old[i, j] + dt * (D * (diff_x + diff_y) - adv_x - adv_y)

# Visualización del estado Final vs Inicial
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(C.T, origin='lower', extent=[0, Lx, 0, Ly], cmap='viridis')
ax1.set_title("Estado Inicial (t=0)")

ax2.imshow(C_new.T, origin='lower', extent=[0, Lx, 0, Ly], cmap='viridis')
ax2.set_title(f"Estado Final tras Advección-Difusión")

plt.suptitle("CFD 2D: Ecuación de Advección-Difusión por Diferencias Finitas")
plt.tight_layout()
plt.show()

print("La mancha ha sido movida físicamente hacia la diagonal (Inercia/Advección) y se ha difuminado ensanchando sus fronteras (Viscosidad/Difusión).")